In [1]:
import pandas as pd
import numpy as np
import json

In [2]:
DATASET = "../data/processed/merged_dataset.parquet"

OUTPUT_JSON = "../data/exports/orbital_uncertainty_clouds.json"

N_ASTEROIDS = 20

N_SIMULATIONS = 250

N_POINTS = 360

In [3]:
df = pd.read_parquet(DATASET)

future_df = df[
    df["is_future"] == True
].copy()

future_df = future_df.sort_values(
    "moid_au"
)

future_df = future_df.head(
    N_ASTEROIDS
)

print(len(future_df))

20


In [4]:
def generate_orbit(
    a,
    e,
    inc_deg,
    points=360
):

    theta = np.linspace(
        0,
        2*np.pi,
        points
    )

    r = (
        a*(1-e**2)
        /
        (1+e*np.cos(theta))
    )

    x = r*np.cos(theta)

    y = r*np.sin(theta)

    inc = np.radians(
        inc_deg
    )

    z = y*np.sin(inc)

    y = y*np.cos(inc)

    return x,y,z

In [5]:
def perturb_orbital_elements(
    a,
    e,
    inc
):

    a_new = np.random.normal(
        a,
        abs(a)*0.01
    )

    e_new = np.random.normal(
        e,
        0.01
    )

    i_new = np.random.normal(
        inc,
        0.5
    )

    e_new = np.clip(
        e_new,
        0,
        0.99
    )

    return (
        a_new,
        e_new,
        i_new
    )

In [6]:
uncertainty_data = []

In [7]:
for _, row in future_df.iterrows():

    asteroid_id = str(
        row["spkid"]
    )

    a = float(row["a"])

    e = float(row["e"])

    inc = float(row["i"])

    simulations = []

    for sim in range(
        N_SIMULATIONS
    ):

        pa, pe, pi = (
            perturb_orbital_elements(
                a,
                e,
                inc
            )
        )

        x,y,z = generate_orbit(
            pa,
            pe,
            pi,
            N_POINTS
        )

        trajectory = []

        for k in range(
            len(x)
        ):

            trajectory.append({

                "x": float(x[k]),

                "y": float(y[k]),

                "z": float(z[k])

            })

        simulations.append({

            "simulation": sim,

            "a": float(pa),

            "e": float(pe),

            "i": float(pi),

            "trajectory":
            trajectory

        })

    uncertainty_data.append({

        "asteroid_id":
        asteroid_id,

        "simulations":
        simulations

    })

In [8]:
with open(
    OUTPUT_JSON,
    "w"
) as f:

    json.dump(
        {
            "asteroids":
            uncertainty_data
        },
        f,
        indent=2
    )

print(
    "Saved:",
    OUTPUT_JSON
)

Saved: ../data/exports/orbital_uncertainty_clouds.json
